# 02 — Genomic Feature Extraction (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleroebijuwa/Explainable-ml-ciprofloxacin-resistance/blob/main/02_feature_extraction_colab.ipynb)

The AMR_ciprofloxacin repository (Pataki et al., 2020) provides MIC values and links to the raw
sequencing reads, but not a ready-to-use feature matrix. This notebook derives genomic features
directly from each isolate's own reads in the European Nucleotide Archive (ENA).

**Features extracted per isolate**

| Feature type | Genes | Method |
|---|---|---|
| Chromosomal amino-acid substitutions | gyrA, gyrB, parC, parE (drug targets); marR, acrR, soxR (efflux regulators) | Reads mapped with BWA-MEM to E. coli K-12 MG1655 genes; consensus base called per position; codons translated and compared with the reference protein |
| Acquired quinolone resistance genes | qnr families, qepA, oqxAB, aac(6')-Ib-cr | KMA against the ResFinder database (same approach as ResFinder, used by Pataki et al.) |
| Lineage / population structure | Whole genome | Mash k-mer sketches → pairwise genomic distances (used later for clade-aware validation) |

**Design decisions**
- Only a subset of read pairs (`N_READ_PAIRS`) is streamed from ENA per isolate. FASTQ files are
  not ordered by genome position, so this gives an even ~30–50× coverage while avoiding downloading
  the full ~3 TB dataset.
- Results for each isolate are saved to Google Drive as soon as they are finished. If the Colab
  session disconnects, simply re-run all cells: finished isolates are skipped.
- Amino-acid positions use E. coli K-12 numbering (e.g. gyrA S83L, parC S80I), the standard
  convention in the resistance literature.

**How to run**
1. `Runtime → Run all`. Allow Google Drive access when asked.
2. Start with the pilot (`N_ISOLATES = 20`) and check the timing and sanity-check output at the end.
3. Set `N_ISOLATES = None` and re-run to process all 704 isolates (several sessions).
4. Download `features.csv`, `qc.csv` and `mash_distances.csv` from `MyDrive/amr_cipro/` into the
   project's `Data/` folder.

## 1. Settings

In [ ]:
# Number of isolates to process. Use a small number for the pilot run,
# then set to None to process every isolate.
N_ISOLATES = 20

# Read pairs streamed per isolate (~30-50x coverage of a 5 Mb E. coli genome)
N_READ_PAIRS = 750_000

# Base calling thresholds for the chromosomal genes
MIN_DEPTH = 5          # minimum reads covering a position
MIN_BASE_FRAC = 0.8    # minimum fraction of reads supporting the majority base
MIN_BASE_QUAL = 20     # ignore bases below this Phred quality
MAX_MISMATCH = 0.10    # discard reads with more than 10% mismatches to the reference gene

# Acquired gene thresholds (ResFinder defaults)
MIN_IDENTITY = 90.0
MIN_COVERAGE = 60.0

RANDOM_SEED = 42
RETRY_ERRORS = False   # set True to re-attempt isolates that failed previously

DRIVE_DIR = "/content/drive/MyDrive/amr_cipro"
DATA_URL = "https://raw.githubusercontent.com/Aleroebijuwa/Explainable-ml-ciprofloxacin-resistance/main/Data"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE = Path(DRIVE_DIR)
RESULTS_DIR = DRIVE / "per_sample"
SKETCH_DIR = DRIVE / "sketches"
for d in [RESULTS_DIR, SKETCH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WORK = Path("/content/work")
REF = Path("/content/reference")
for d in [WORK, REF]:
    d.mkdir(parents=True, exist_ok=True)

print("Results will be saved to:", DRIVE)

## 3. Install bioinformatics tools
- **BWA** and **SAMtools**: read mapping and BAM handling
- **KMA**: k-mer alignment used by ResFinder for acquired genes
- **Mash**: genome distance estimation
- **pysam**, **Biopython**: reading alignments and translating sequences in Python

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y bwa samtools > /dev/null
pip -q install pysam biopython

if ! command -v kma > /dev/null; then
    git clone -q --depth 1 https://bitbucket.org/genomicepidemiology/kma.git /content/kma
    make -s -C /content/kma > /dev/null 2>&1
    cp /content/kma/kma /content/kma/kma_index /usr/local/bin/
fi

if ! command -v mash > /dev/null; then
    wget -q -O /tmp/mash.tar https://github.com/marbl/Mash/releases/download/v2.3/mash-Linux64-v2.3.tar
    tar -xf /tmp/mash.tar -C /tmp
    cp /tmp/mash-Linux64-v2.3/mash /usr/local/bin/
fi

echo "Tool versions (record these in the methods section):"
bwa 2>&1 | grep Version
samtools --version | head -1
echo "kma $(kma -v 2>&1 | head -1)"
echo "mash $(mash --version)"

In [ ]:
import io
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
import pysam
import requests
from Bio import SeqIO
from Bio.Seq import Seq

THREADS = os.cpu_count()
print("CPUs available:", THREADS)
print("pysam", pysam.__version__)

## 4. Load the isolate list
Both metadata files are read from the project GitHub repository. Each isolate is linked to its
paired-end FASTQ files through `sample_alias`. The isolate order is shuffled with a fixed seed so the
pilot run is a random sample of the dataset.

In [ ]:
metadata = pd.read_csv(f"{DATA_URL}/supplementary_meta_table.csv")
sequencing_metadata = pd.read_csv(f"{DATA_URL}/meta.tsv", sep="\t")

isolates = metadata.merge(
    sequencing_metadata[["sample_alias", "fastq_ftp"]], on="sample_alias", how="left"
)

def paired_fastq_urls(fastq_ftp):
    # Return the first _1/_2 FASTQ pair as HTTPS links
    files = str(fastq_ftp).split(";")
    r1 = [f for f in files if f.endswith("_1.fastq.gz")]
    r2 = [f for f in files if f.endswith("_2.fastq.gz")]
    if not r1 or not r2:
        return None, None
    return "https://" + r1[0], "https://" + r2[0]

isolates[["url_r1", "url_r2"]] = isolates["fastq_ftp"].apply(
    lambda s: pd.Series(paired_fastq_urls(s))
)
isolates["run_accession"] = isolates["url_r1"].str.extract(r"/([EDS]RR\d+)_1")[0]
isolates["sample_id"] = isolates["sample_alias"].str.replace(r"[^A-Za-z0-9_.-]", "_", regex=True)

print("Isolates:", len(isolates))
print("Isolates with paired FASTQ links:", isolates["url_r1"].notna().sum())
assert isolates["sample_id"].is_unique

isolates = isolates.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

## 5. Build the reference sequences

**Chromosomal targets.** Gene sequences are taken from the annotated *E. coli* K-12 MG1655 genome
(NCBI RefSeq NC_000913.3). A 150 bp flank is added on each side so that reads overlapping the gene
ends still map.

**Acquired genes.** The ResFinder quinolone database is combined with the aac(6')-Ib entries from the
aminoglycoside database. The wild-type aac(6')-Ib alleles are included so that KMA can tell them apart
from the ciprofloxacin-modifying aac(6')-Ib-cr variant (W102R, D179Y).

In [ ]:
TARGET_GENES = ["gyrA", "gyrB", "parC", "parE", "marR", "acrR", "soxR"]
FLANK = 150

genbank_url = ("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
               "?db=nuccore&id=NC_000913.3&rettype=gbwithparts&retmode=text")
mg1655 = SeqIO.read(io.StringIO(requests.get(genbank_url, timeout=300).text), "genbank")

chrom_targets = {}
with open(REF / "chromosomal_targets.fa", "w") as fa:
    for feature in mg1655.features:
        gene = feature.qualifiers.get("gene", [None])[0]
        if feature.type != "CDS" or gene not in TARGET_GENES:
            continue
        start, end = int(feature.location.start), int(feature.location.end)
        region = mg1655.seq[start - FLANK:end + FLANK]
        if feature.location.strand == -1:
            region = region.reverse_complement()
        cds = str(region[FLANK:FLANK + (end - start)])

        # Check the extracted CDS translates to the annotated protein
        assert str(Seq(cds).translate(table=11, cds=True)) == feature.qualifiers["translation"][0]

        chrom_targets[gene] = {"cds": cds, "contig_length": len(region)}
        fa.write(f">{gene}\n{region}\n")

missing = set(TARGET_GENES) - set(chrom_targets)
assert not missing, f"Genes not found: {missing}"
for gene, info in chrom_targets.items():
    print(f"{gene}: {len(info['cds'])} bp, {len(info['cds']) // 3 - 1} amino acids")

In [ ]:
RESFINDER = "https://bitbucket.org/genomicepidemiology/resfinder_db/raw/master"

def read_fasta_text(text):
    records, name = {}, None
    for line in text.splitlines():
        if line.startswith(">"):
            name = line[1:].strip()
            records[name] = []
        elif name:
            records[name].append(line.strip())
    return {k: "".join(v) for k, v in records.items()}

quinolone_db = read_fasta_text(requests.get(f"{RESFINDER}/quinolone.fsa", timeout=300).text)
aminoglycoside_db = read_fasta_text(requests.get(f"{RESFINDER}/aminoglycoside.fsa", timeout=300).text)
aac6_ib = {k: v for k, v in aminoglycoside_db.items() if re.match(r"aac\(6'\)-?Ib(-cr)?_\d", k)}

with open(REF / "acquired_genes.fa", "w") as fa:
    for name, seq in {**quinolone_db, **aac6_ib}.items():
        fa.write(f">{name}\n{seq}\n")

print("Quinolone resistance alleles:", len(quinolone_db))
print("aac(6')-Ib alleles:", list(aac6_ib))

In [ ]:
%%bash
cd /content/reference
bwa index chromosomal_targets.fa 2> /dev/null
kma_index -i acquired_genes.fa -o acquired_genes > /dev/null 2>&1
ls -la

## 6. Feature extraction functions

In [ ]:
def sh(cmd):
    # Run a bash command and raise an error with its output if it fails
    result = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{result.stderr[-2000:]}")
    return result.stdout


def stream_reads(url, out_path, n_pairs):
    # Download only the first n_pairs reads; head stops the download early
    sh(f"set +o pipefail; curl -sL --retry 3 --max-time 1800 '{url}' "
       f"| gzip -dc 2>/dev/null | head -n {4 * n_pairs} > '{out_path}'")
    return int(sh(f"wc -l < '{out_path}'")) // 4


def good_read(read):
    # Keep primary alignments with few mismatches to the reference gene
    if read.is_unmapped or read.is_secondary or read.is_supplementary or read.is_qcfail:
        return False
    aligned = read.query_alignment_length
    return aligned >= 50 and read.get_tag("NM") <= MAX_MISMATCH * aligned


def consensus_sequence(bam_path, contig, length):
    # Majority base at each position; 'N' where depth or agreement is too low
    with pysam.AlignmentFile(bam_path) as bam:
        counts = np.array(bam.count_coverage(
            contig, 0, length, quality_threshold=MIN_BASE_QUAL, read_callback=good_read
        ))
    depth = counts.sum(axis=0)
    majority = counts.max(axis=0)
    agreement = np.divide(majority, depth, out=np.zeros(length), where=depth > 0)
    bases = np.array(list("ACGT"))[counts.argmax(axis=0)]
    bases[(depth < MIN_DEPTH) | (agreement < MIN_BASE_FRAC)] = "N"
    return "".join(bases), depth


def amino_acid_changes(gene, consensus_cds, reference_cds):
    # Compare translated proteins; codons containing N are treated as unknown (X)
    ref_protein = str(Seq(reference_cds).translate(table=11))
    sample_protein = str(Seq(consensus_cds).translate(table=11))
    changes = []
    for position, (ref_aa, sample_aa) in enumerate(zip(ref_protein, sample_protein), start=1):
        if sample_aa not in ("X", ref_aa):
            changes.append(f"{gene}_{ref_aa}{position}{sample_aa}".replace("*", "stop"))
    return changes


def chromosomal_features(r1, r2, work_dir):
    bam = work_dir / "chromosomal.bam"
    sh(f"bwa mem -t {THREADS} -v 1 {REF}/chromosomal_targets.fa '{r1}' '{r2}' 2> /dev/null "
       f"| samtools sort -@ {THREADS} -o '{bam}' - && samtools index '{bam}'")
    genes = {}
    for gene, info in chrom_targets.items():
        consensus, depth = consensus_sequence(str(bam), gene, info["contig_length"])
        cds_consensus = consensus[FLANK:FLANK + len(info["cds"])]
        cds_depth = depth[FLANK:FLANK + len(info["cds"])]
        genes[gene] = {
            "breadth": round(float((cds_depth >= MIN_DEPTH).mean()), 4),
            "mean_depth": round(float(cds_depth.mean()), 1),
            "changes": amino_acid_changes(gene, cds_consensus, info["cds"]),
        }
    return genes


def gene_family(template):
    name = template.split()[0]
    if "Ib-cr" in name:
        return "aac(6')-Ib-cr"
    if name.startswith("aac(6')"):
        return None  # wild-type aac(6')-Ib: aminoglycoside gene, no quinolone activity
    match = re.match(r"(qnr[A-Z]+|qepA|[Oo]qx[AB])", name)
    return match.group(1).replace("Oqx", "oqx") if match else name.split("_")[0]


def acquired_features(r1, r2, work_dir):
    prefix = work_dir / "kma"
    sh(f"kma -ipe '{r1}' '{r2}' -o '{prefix}' -t_db {REF}/acquired_genes -1t1 -t {THREADS} > /dev/null 2>&1")
    res = pd.read_csv(f"{prefix}.res", sep="\t")
    res.columns = [c.strip().lstrip("#") for c in res.columns]
    hits = []
    for _, row in res.iterrows():
        family = gene_family(row["Template"])
        if family and row["Template_Identity"] >= MIN_IDENTITY and row["Template_Coverage"] >= MIN_COVERAGE:
            hits.append({
                "family": family,
                "template": row["Template"].split()[0],
                "identity": float(row["Template_Identity"]),
                "coverage": float(row["Template_Coverage"]),
                "depth": float(row["Depth"]),
            })
    return hits


def mash_sketch(r1, r2, sample_id, work_dir):
    combined = work_dir / "combined.fq"
    sh(f"cat '{r1}' '{r2}' > '{combined}'")
    sh(f"mash sketch -r -m 2 -k 21 -s 10000 -I '{sample_id}' -o '{SKETCH_DIR / sample_id}' '{combined}'")

In [ ]:
def process_isolate(row):
    start_time = time.time()
    work_dir = WORK / row.sample_id
    work_dir.mkdir(parents=True, exist_ok=True)
    r1, r2 = work_dir / "r1.fq", work_dir / "r2.fq"
    try:
        n1 = stream_reads(row.url_r1, r1, N_READ_PAIRS)
        n2 = stream_reads(row.url_r2, r2, N_READ_PAIRS)
        n_pairs = min(n1, n2)
        if n_pairs < 50_000:
            raise RuntimeError(f"Too few reads downloaded ({n1}, {n2})")
        if n1 != n2:  # keep the read files in sync
            for path, n in [(r1, n1), (r2, n2)]:
                if n > n_pairs:
                    sh(f"head -n {4 * n_pairs} '{path}' > '{path}.tmp' && mv '{path}.tmp' '{path}'")

        read_length = len(sh(f"sed -n 2p '{r1}'").strip())
        result = {
            "sample_alias": row.sample_alias,
            "sample_id": row.sample_id,
            "run_accession": row.run_accession,
            "status": "ok",
            "n_read_pairs": n_pairs,
            "read_length": read_length,
            "estimated_coverage": round(2 * n_pairs * read_length / 5.0e6, 1),
            "genes": chromosomal_features(r1, r2, work_dir),
            "acquired": acquired_features(r1, r2, work_dir),
        }
        mash_sketch(r1, r2, row.sample_id, work_dir)
    except Exception as error:
        result = {"sample_alias": row.sample_alias, "sample_id": row.sample_id,
                  "status": "error", "message": str(error)[-1500:]}
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

    result["seconds"] = round(time.time() - start_time, 1)
    # Write to a temporary file first so a disconnect never leaves a half-written result
    tmp_path = RESULTS_DIR / f"{row.sample_id}.json.tmp"
    tmp_path.write_text(json.dumps(result, indent=1))
    tmp_path.replace(RESULTS_DIR / f"{row.sample_id}.json")
    return result

## 7. Run the extraction
Finished isolates are skipped, so this cell can be re-run after a disconnect.

In [ ]:
def already_done(sample_id):
    path = RESULTS_DIR / f"{sample_id}.json"
    if not path.exists():
        return False
    return json.loads(path.read_text())["status"] == "ok" or not RETRY_ERRORS

to_process = isolates if N_ISOLATES is None else isolates.head(N_ISOLATES)
to_process = to_process[to_process["url_r1"].notna()]
pending = [row for row in to_process.itertuples() if not already_done(row.sample_id)]
print(f"{len(to_process) - len(pending)} already done, {len(pending)} to process\n")

run_start = time.time()
for i, row in enumerate(pending, start=1):
    result = process_isolate(row)
    if result["status"] == "ok":
        mutations = [c for g in result["genes"].values() for c in g["changes"]
                     if c.split("_")[0] in ("gyrA", "parC", "parE")]
        acquired = sorted({h["family"] for h in result["acquired"]})
        summary = (f"cov {result['estimated_coverage']}x | "
                   f"QRDR: {', '.join(mutations) or '-'} | acquired: {', '.join(acquired) or '-'}")
    else:
        summary = "ERROR: " + result["message"].splitlines()[0][:150]
    elapsed = (time.time() - run_start) / 60
    print(f"[{i}/{len(pending)}] {row.sample_alias} ({result['seconds']:.0f}s, {elapsed:.0f} min total) {summary}")

## 8. Combine results into feature tables
- `features.csv`: one row per isolate, one binary column per observed amino-acid substitution
  (e.g. `gyrA_S83L`) and per acquired gene family (e.g. `acq_qnrS`).
- `qc.csv`: read counts, coverage and per-gene breadth, used to exclude poor-quality isolates.

No features are filtered here. Rare-feature filtering is done later inside the training folds to avoid
data leakage.

In [ ]:
results = [json.loads(p.read_text()) for p in sorted(RESULTS_DIR.glob("*.json"))]
ok_results = [r for r in results if r["status"] == "ok"]
failed = [r for r in results if r["status"] == "error"]
print(f"Successful: {len(ok_results)}, failed: {len(failed)}")
for r in failed:
    print("  ", r["sample_alias"], "->", r["message"].splitlines()[0][:150])

feature_rows, qc_rows = [], []
for r in ok_results:
    features = {"sample_alias": r["sample_alias"]}
    for gene in r["genes"].values():
        for change in gene["changes"]:
            features[change] = 1
    for hit in r["acquired"]:
        features[f"acq_{hit['family']}"] = 1
    feature_rows.append(features)

    qc = {k: r[k] for k in ["sample_alias", "run_accession", "n_read_pairs",
                            "read_length", "estimated_coverage", "seconds"]}
    for gene_name, gene in r["genes"].items():
        qc[f"{gene_name}_breadth"] = gene["breadth"]
        qc[f"{gene_name}_depth"] = gene["mean_depth"]
    qc["acquired_alleles"] = ";".join(h["template"] for h in r["acquired"])
    qc_rows.append(qc)

features = pd.DataFrame(feature_rows).fillna(0)
feature_cols = sorted(c for c in features.columns if c != "sample_alias")
features[feature_cols] = features[feature_cols].astype(int)
features = features[["sample_alias"] + feature_cols]
qc = pd.DataFrame(qc_rows)

features.to_csv(DRIVE / "features.csv", index=False)
qc.to_csv(DRIVE / "qc.csv", index=False)
print("features.csv:", features.shape)
print("qc.csv:", qc.shape)

In [ ]:
# Pairwise Mash distances between all processed isolates (used for lineage clustering later)
ok_ids = [r["sample_id"] for r in ok_results]
(WORK / "sketch_list.txt").write_text("\n".join(str(SKETCH_DIR / f"{i}.msh") for i in ok_ids))
sh(f"rm -f {WORK}/all_sketches.msh && mash paste {WORK}/all_sketches -l {WORK}/sketch_list.txt")
dist_text = sh(f"mash dist -p {THREADS} -t {WORK}/all_sketches.msh {WORK}/all_sketches.msh")

# Mash labels each isolate with the sample_id given to -I when sketching
distances = pd.read_csv(io.StringIO(dist_text), sep="\t", index_col=0)
distances.columns = distances.index
id_to_alias = dict(zip(isolates["sample_id"], isolates["sample_alias"]))
distances = distances.rename(index=id_to_alias, columns=id_to_alias)
distances.index.name = "sample_alias"
distances.to_csv(DRIVE / "mash_distances.csv")
print("mash_distances.csv:", distances.shape)

## 9. Sanity check
If the extraction worked, the well-known resistance mutations (gyrA S83L, gyrA D87N, parC S80I)
should be far more common in resistant isolates. Resistance is defined here with the EUCAST clinical
breakpoint (MIC > 0.5 mg/L) for this check only; the label is defined formally in the next notebook.

In [ ]:
check = metadata[["sample_alias", "measurement"]].merge(features, on="sample_alias")
check["resistant"] = (check["measurement"] > 0.5).astype(int)
print(f"Isolates checked: {len(check)}, resistant: {check['resistant'].sum()}\n")

key_features = ["gyrA_S83L", "gyrA_D87N", "parC_S80I", "parC_E84V", "parE_S458A", "acq_qnrS", "acq_aac(6')-Ib-cr"]
summary = pd.DataFrame({
    f: {
        "% of resistant isolates": round(100 * check.loc[check.resistant == 1, f].mean(), 1) if f in check else 0.0,
        "% of susceptible isolates": round(100 * check.loc[check.resistant == 0, f].mean(), 1) if f in check else 0.0,
    }
    for f in key_features
}).T
display(summary)

minutes = qc["seconds"].mean() / 60
print(f"\nAverage time per isolate: {minutes:.1f} min")
print(f"Estimated time for all {len(isolates)} isolates: {minutes * len(isolates) / 60:.1f} hours")
print(f"Median estimated coverage: {qc['estimated_coverage'].median():.0f}x")
print("Isolates with <95% breadth in gyrA or parC:",
      int(((qc["gyrA_breadth"] < 0.95) | (qc["parC_breadth"] < 0.95)).sum()))